In [ ]:
# =============================================================================
# Leuven Carrier-Receiver Collaboration Analysis
# =============================================================================
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gzip
from lxml import etree
import matsim
from shapely.geometry import Point, LineString

# Configuration and Paths

In [ ]:
# Base output path for Leuven Carrier-Receiver Collaboration
leuven_output_path = r'output/leuvenCarrierReceiverCollab/'

# Network path
network_path = r'data/GemeenteLeuvenWithHbefaType/GemeenteLeuvenWithHbefaType.xml.gz'

# Folder naming pattern: leuvenCRCollab-{depot_location}-{distribution}-{method}-af{af}-p{penalty}-i{instance}
# depot_location: 'inside' or 'outside' (the study area)
# distribution: 'clustered' or 'dispersed'
# method: 'approx_shapley_mc', 'approx_shapley_stratified', 'marginal', 'proportional'
# af: allocation factor (e.g., 0.80)
# penalty: penalty value (e.g., 0.0000)
# instance: instance number (e.g., i01)

# Get all folders
all_leuven_folders = [f for f in os.listdir(leuven_output_path) if os.path.isdir(os.path.join(leuven_output_path, f))]
print(f"Total folders: {len(all_leuven_folders)}")
print(f"Sample folders: {all_leuven_folders[:5]}")

# Read Network using matsim package

In [ ]:
# Read network using matsim package - returns a GeoDataFrame with links as LineStrings
network = matsim.read_network(network_path)
network_gdf = network.as_geo()

print(f"Network contains {len(network_gdf)} links")
network_gdf.head()

In [ ]:
# Visualize the network
fig, ax = plt.subplots(figsize=(12, 12))
network_gdf.plot(ax=ax, color='lightgrey', linewidth=0.3)
ax.set_title('Leuven Network')
plt.tight_layout()
plt.show()

# Core Functions: Parse Folder Name and Read XML Files

In [ ]:
# =============================================================================
# Core Parsing Functions
# =============================================================================

def parse_leuven_folder_name(folder_name):
    """
    Parse the Leuven folder naming pattern.
    Pattern: leuvenCRCollab-{depot_location}-{distribution}-{method}-af{af}-p{penalty}-i{instance}
    
    Example: leuvenCRCollab-inside-clustered-approx_shapley_mc-af0.80-p0.0000-i01
    
    Returns:
    --------
    dict : Dictionary with keys 'depot_location', 'distribution', 'method', 'af', 'penalty', 'instance'
    """
    parts = folder_name.split('-')
    
    # Extract components based on position
    depot_location = parts[1]  # 'inside' or 'outside'
    distribution = parts[2]     # 'clustered' or 'dispersed'
    
    # Method can have underscores, need to find where af starts
    af_idx = None
    for i, part in enumerate(parts):
        if part.startswith('af'):
            af_idx = i
            break
    
    method = '-'.join(parts[3:af_idx]) if af_idx else parts[3]
    af = parts[af_idx].replace('af', '') if af_idx else None
    penalty = parts[af_idx + 1].replace('p', '') if af_idx else None
    instance = parts[af_idx + 2] if af_idx else None
    
    return {
        'depot_location': depot_location,
        'distribution': distribution,
        'method': method,
        'af': af,
        'penalty': penalty,
        'instance': instance
    }


def read_receivers_from_xml(receiver_file_path):
    """
    Read receiver information from receivers.xml.gz file.
    
    Returns:
    --------
    DataFrame with columns: receiver_id, link_id, affiliated_carrier_id, 
                           collaboration_status, grand_coalition_member, score,
                           selected_plan_score, time_window_start, time_window_end, order_carrier_id
    """
    receivers_data = []
    
    with gzip.open(receiver_file_path, 'rb') as f:
        tree = etree.parse(f)
        root = tree.getroot()
    
    for receiver in root.findall('.//receiver'):
        receiver_id = receiver.get('id')
        link_id = receiver.get('linkId')
        
        # Parse attributes
        attributes = {}
        for attr in receiver.findall('.//attribute'):
            attr_name = attr.get('name')
            attr_value = attr.text
            # Convert to appropriate type
            attr_class = attr.get('class', '')
            if 'Boolean' in attr_class:
                attr_value = attr_value.lower() == 'true'
            elif 'Double' in attr_class:
                attr_value = float(attr_value)
            attributes[attr_name] = attr_value
        
        # Get selected plan info
        selected_plan = receiver.find(".//plan[@selected='yes']")
        selected_plan_score = None
        time_window_start = None
        time_window_end = None
        order_carrier_id = None
        
        if selected_plan is not None:
            selected_plan_score = float(selected_plan.get('score', -100))
            time_window = selected_plan.find('timeWindow')
            if time_window is not None:
                time_window_start = time_window.get('start')
                time_window_end = time_window.get('end')
            order = selected_plan.find('order')
            if order is not None:
                order_carrier_id = order.get('carrierId')
        
        receivers_data.append({
            'receiver_id': receiver_id,
            'link_id': link_id,
            'affiliated_carrier_id': attributes.get('affiliatedCarrierId'),
            'collaboration_status': attributes.get('collaborationStatus', False),
            'grand_coalition_member': attributes.get('grandCoalitionMember', False),
            'score': attributes.get('score', -100),
            'selected_plan_score': selected_plan_score,
            'time_window_start': time_window_start,
            'time_window_end': time_window_end,
            'order_carrier_id': order_carrier_id
        })
    
    return pd.DataFrame(receivers_data)


def read_carriers_from_xml(carrier_file_path):
    """
    Read carrier information from carriers.xml.gz file.
    
    Returns:
    --------
    DataFrame with columns: carrier_id, depot_link_id, vehicle_ids, jsprit_score, num_shipments
    """
    carriers_data = []
    
    with gzip.open(carrier_file_path, 'rb') as f:
        tree = etree.parse(f)
        root = tree.getroot()
    
    # Handle namespace
    ns = {'m': 'http://www.matsim.org/files/dtd'}
    
    for carrier in root.findall('.//{http://www.matsim.org/files/dtd}carrier'):
        carrier_id = carrier.get('id')
        
        # Get depot link IDs from vehicles
        depot_links = set()
        vehicle_ids = []
        for vehicle in carrier.findall('.//{http://www.matsim.org/files/dtd}vehicle'):
            depot_link_id = vehicle.get('depotLinkId')
            vehicle_id = vehicle.get('id')
            if depot_link_id:
                depot_links.add(depot_link_id)
            if vehicle_id:
                vehicle_ids.append(vehicle_id)
        
        # Count shipments
        shipments = carrier.findall('.//{http://www.matsim.org/files/dtd}shipment')
        num_shipments = len(shipments)
        
        # Get jsprit score from selected plan
        jsprit_score = None
        selected_plan = carrier.find('.//{http://www.matsim.org/files/dtd}plan[@selected="true"]')
        if selected_plan is not None:
            for attr in selected_plan.findall('.//{http://www.matsim.org/files/dtd}attribute'):
                if attr.get('name') == 'jspritScore':
                    jsprit_score = float(attr.text)
        
        carriers_data.append({
            'carrier_id': carrier_id,
            'depot_link_ids': list(depot_links),
            'depot_link_id': list(depot_links)[0] if depot_links else None,  # Primary depot
            'vehicle_ids': vehicle_ids,
            'num_vehicles': len(vehicle_ids),
            'jsprit_score': jsprit_score,
            'num_shipments': num_shipments
        })
    
    return pd.DataFrame(carriers_data)


def read_receiver_stats_csv(stats_file_path):
    """
    Read receiver_stats.csv file which contains iteration-level receiver data.
    
    Returns:
    --------
    DataFrame with all receiver stats
    """
    return pd.read_csv(stats_file_path)


# Test the parsing function
sample_folder = all_leuven_folders[0]
print(f"Sample folder: {sample_folder}")
print(f"Parsed: {parse_leuven_folder_name(sample_folder)}")

# Functions: Create GeoDataFrames for Receivers and Carriers

In [ ]:
# =============================================================================
# GeoDataFrame Creation Functions
# =============================================================================

def get_link_centroid(link_id, network_gdf):
    """
    Get the centroid of a link from the network GeoDataFrame.
    
    Parameters:
    -----------
    link_id : str
        The link ID
    network_gdf : GeoDataFrame
        Network GeoDataFrame from matsim.read_network().as_geo()
    
    Returns:
    --------
    Point : Shapely Point at the centroid of the link, or None if not found
    """
    if link_id in network_gdf.index:
        return network_gdf.loc[link_id].geometry.centroid
    return None


def create_receiver_gdf(receiver_df, network_gdf):
    """
    Create a GeoDataFrame of receivers with Point geometry based on link centroids.
    
    Parameters:
    -----------
    receiver_df : DataFrame
        DataFrame from read_receivers_from_xml()
    network_gdf : GeoDataFrame
        Network GeoDataFrame
    
    Returns:
    --------
    GeoDataFrame with receiver information and Point geometry
    """
    geometries = []
    for _, row in receiver_df.iterrows():
        geom = get_link_centroid(row['link_id'], network_gdf)
        geometries.append(geom)
    
    receiver_gdf = gpd.GeoDataFrame(receiver_df, geometry=geometries, crs=network_gdf.crs)
    # Remove rows with no geometry
    receiver_gdf = receiver_gdf[receiver_gdf.geometry.notna()]
    return receiver_gdf


def create_carrier_gdf(carrier_df, network_gdf):
    """
    Create a GeoDataFrame of carriers (depot locations) with Point geometry.
    
    Parameters:
    -----------
    carrier_df : DataFrame
        DataFrame from read_carriers_from_xml()
    network_gdf : GeoDataFrame
        Network GeoDataFrame
    
    Returns:
    --------
    GeoDataFrame with carrier information and Point geometry (depot location)
    """
    geometries = []
    for _, row in carrier_df.iterrows():
        geom = get_link_centroid(row['depot_link_id'], network_gdf)
        geometries.append(geom)
    
    carrier_gdf = gpd.GeoDataFrame(carrier_df, geometry=geometries, crs=network_gdf.crs)
    carrier_gdf = carrier_gdf[carrier_gdf.geometry.notna()]
    return carrier_gdf


def load_scenario_data(output_path, folder_name, network_gdf):
    """
    Load all data for a single scenario folder and return GeoDataFrames.
    
    Parameters:
    -----------
    output_path : str
        Base output path
    folder_name : str
        Folder name
    network_gdf : GeoDataFrame
        Network GeoDataFrame
    
    Returns:
    --------
    dict : Dictionary containing:
        - 'scenario_info': parsed folder name info
        - 'receiver_df': raw receiver DataFrame
        - 'receiver_gdf': receiver GeoDataFrame
        - 'carrier_df': raw carrier DataFrame
        - 'carrier_gdf': carrier GeoDataFrame
        - 'receiver_stats': receiver stats from CSV
    """
    folder_path = os.path.join(output_path, folder_name)
    
    # Parse folder name
    scenario_info = parse_leuven_folder_name(folder_name)
    
    # Read receivers
    receiver_file = os.path.join(folder_path, 'receivers.xml.gz')
    receiver_df = read_receivers_from_xml(receiver_file)
    receiver_gdf = create_receiver_gdf(receiver_df, network_gdf)
    
    # Read carriers
    carrier_file = os.path.join(folder_path, 'carriers.xml.gz')
    carrier_df = read_carriers_from_xml(carrier_file)
    carrier_gdf = create_carrier_gdf(carrier_df, network_gdf)
    
    # Read receiver stats
    stats_file = os.path.join(folder_path, 'receiver_stats.csv')
    receiver_stats = read_receiver_stats_csv(stats_file) if os.path.exists(stats_file) else None
    
    return {
        'scenario_info': scenario_info,
        'folder_name': folder_name,
        'receiver_df': receiver_df,
        'receiver_gdf': receiver_gdf,
        'carrier_df': carrier_df,
        'carrier_gdf': carrier_gdf,
        'receiver_stats': receiver_stats
    }


# Test loading a single scenario
sample_data = load_scenario_data(leuven_output_path, sample_folder, network_gdf)
print(f"Loaded scenario: {sample_data['scenario_info']}")
print(f"Receivers: {len(sample_data['receiver_gdf'])}")
print(f"Carriers: {len(sample_data['carrier_gdf'])}")

In [ ]:
# Display sample receiver GeoDataFrame
sample_data['receiver_gdf'].head(10)

In [ ]:
# Display sample carrier GeoDataFrame
sample_data['carrier_gdf']

# Visualization Functions

In [ ]:
# =============================================================================
# Visualization Functions
# =============================================================================

def plot_scenario_map(scenario_data, network_gdf, bbox=None, figsize=(12, 12), 
                      show_collaboration=True, title=None):
    """
    Plot a map showing network, receivers, and carriers for a scenario.
    
    Parameters:
    -----------
    scenario_data : dict
        Output from load_scenario_data()
    network_gdf : GeoDataFrame
        Network GeoDataFrame
    bbox : tuple, optional
        Bounding box (xmin, xmax, ymin, ymax) to crop the view
    figsize : tuple
        Figure size
    show_collaboration : bool
        If True, color receivers by collaboration status
    title : str, optional
        Custom title
    
    Returns:
    --------
    matplotlib.figure.Figure
    """
    receiver_gdf = scenario_data['receiver_gdf']
    carrier_gdf = scenario_data['carrier_gdf']
    info = scenario_data['scenario_info']
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Crop network if bbox provided
    if bbox:
        network_plot = network_gdf.cx[bbox[0]:bbox[1], bbox[2]:bbox[3]]
        receiver_plot = receiver_gdf.cx[bbox[0]:bbox[1], bbox[2]:bbox[3]]
        carrier_plot = carrier_gdf.cx[bbox[0]:bbox[1], bbox[2]:bbox[3]]
    else:
        network_plot = network_gdf
        receiver_plot = receiver_gdf
        carrier_plot = carrier_gdf
    
    # Plot network
    network_plot.plot(ax=ax, color='lightgrey', linewidth=0.3, alpha=0.5)
    
    # Plot receivers
    if show_collaboration and 'collaboration_status' in receiver_plot.columns:
        # Color by collaboration status
        collab_receivers = receiver_plot[receiver_plot['collaboration_status'] == True]
        non_collab_receivers = receiver_plot[receiver_plot['collaboration_status'] == False]
        
        if len(non_collab_receivers) > 0:
            non_collab_receivers.plot(ax=ax, color='red', markersize=30, marker='x', 
                                      label=f'Non-collaborative ({len(non_collab_receivers)})', zorder=3)
        if len(collab_receivers) > 0:
            collab_receivers.plot(ax=ax, color='blue', markersize=30, marker='o', 
                                  label=f'Collaborative ({len(collab_receivers)})', alpha=0.7, zorder=3)
    else:
        receiver_plot.plot(ax=ax, color='blue', markersize=30, marker='o', 
                          label=f'Receivers ({len(receiver_plot)})', alpha=0.7, zorder=3)
    
    # Plot carriers (depots)
    carrier_plot.plot(ax=ax, color='red', markersize=150, marker='*', 
                      label=f'Carrier Depots ({len(carrier_plot)})', zorder=4, edgecolor='darkred')
    
    if title is None:
        title = f"Leuven CR Collaboration\n{info['depot_location'].title()} | {info['distribution'].title()} | {info['method']}\npenalty={info['penalty']}"
    
    ax.set_title(title, fontsize=12)
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    
    plt.tight_layout()
    plt.show()
    return fig


def plot_receiver_scores_on_map(scenario_data, network_gdf, bbox=None, figsize=(12, 12)):
    """
    Plot receivers colored by their score.
    """
    receiver_gdf = scenario_data['receiver_gdf']
    carrier_gdf = scenario_data['carrier_gdf']
    info = scenario_data['scenario_info']
    
    fig, ax = plt.subplots(figsize=figsize)
    
    if bbox:
        network_plot = network_gdf.cx[bbox[0]:bbox[1], bbox[2]:bbox[3]]
        receiver_plot = receiver_gdf.cx[bbox[0]:bbox[1], bbox[2]:bbox[3]]
        carrier_plot = carrier_gdf.cx[bbox[0]:bbox[1], bbox[2]:bbox[3]]
    else:
        network_plot = network_gdf
        receiver_plot = receiver_gdf
        carrier_plot = carrier_gdf
    
    # Plot network
    network_plot.plot(ax=ax, color='lightgrey', linewidth=0.3, alpha=0.5)
    
    # Plot receivers colored by score
    # Filter out -100 scores (these are initialization scores)
    receiver_plot_valid = receiver_plot[receiver_plot['score'] > -100]
    
    if len(receiver_plot_valid) > 0:
        receiver_plot_valid.plot(ax=ax, column='score', cmap='RdYlGn', markersize=40, 
                                  legend=True, legend_kwds={'label': 'Receiver Score'},
                                  alpha=0.8, zorder=3)
    
    # Plot carriers
    carrier_plot.plot(ax=ax, color='red', markersize=150, marker='*', 
                      label='Carrier Depots', zorder=4, edgecolor='darkred')
    
    title = f"Receiver Scores\n{info['depot_location'].title()} | {info['distribution'].title()} | p={info['penalty']}"
    ax.set_title(title, fontsize=12)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()
    return fig


def compare_distributions(output_path, depot_location='inside', penalty='0.0000', 
                          method='approx_shapley_mc', network_gdf=None, bbox=None,
                          figsize=(16, 7)):
    """
    Compare clustered vs dispersed distributions side by side.
    """
    all_folders = os.listdir(output_path)
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    for idx, distribution in enumerate(['clustered', 'dispersed']):
        ax = axes[idx]
        
        # Find matching folder
        pattern = f'leuvenCRCollab-{depot_location}-{distribution}-{method}-af0.80-p{penalty}'
        matching = [f for f in all_folders if f.startswith(pattern)]
        
        if not matching:
            ax.text(0.5, 0.5, f'No data for {distribution}', ha='center', va='center')
            continue
        
        folder = matching[0]
        data = load_scenario_data(output_path, folder, network_gdf)
        
        receiver_gdf = data['receiver_gdf']
        carrier_gdf = data['carrier_gdf']
        
        if bbox:
            network_plot = network_gdf.cx[bbox[0]:bbox[1], bbox[2]:bbox[3]]
            receiver_plot = receiver_gdf.cx[bbox[0]:bbox[1], bbox[2]:bbox[3]]
            carrier_plot = carrier_gdf.cx[bbox[0]:bbox[1], bbox[2]:bbox[3]]
        else:
            network_plot = network_gdf
            receiver_plot = receiver_gdf
            carrier_plot = carrier_gdf
        
        # Plot
        network_plot.plot(ax=ax, color='lightgrey', linewidth=0.3, alpha=0.5)
        
        collab = receiver_plot[receiver_plot['collaboration_status'] == True]
        non_collab = receiver_plot[receiver_plot['collaboration_status'] == False]
        
        if len(non_collab) > 0:
            non_collab.plot(ax=ax, color='red', markersize=25, marker='x', zorder=3,
                           label=f'Non-collab ({len(non_collab)})')
        if len(collab) > 0:
            collab.plot(ax=ax, color='blue', markersize=25, marker='o', alpha=0.7, zorder=3,
                       label=f'Collab ({len(collab)})')
        
        carrier_plot.plot(ax=ax, color='red', markersize=120, marker='*', zorder=4, 
                         edgecolor='darkred', label='Depot')
        
        ax.set_title(f'{distribution.title()} Distribution', fontsize=12, fontweight='bold')
        ax.legend(loc='upper right', fontsize=8)
        ax.set_aspect('equal')
    
    fig.suptitle(f'{depot_location.title()} Depot | {method} | penalty={penalty}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    return fig


print("Visualization functions defined.")

# Example Visualizations

In [ ]:
# Define bounding box for the study area (based on leuven_poi.ipynb)
# Visualization bbox: (Xmin:171161.734, Xmax:175265.856, Ymin:172412.828, Ymax:176444.093)
bbox_vis = (171161.734, 175265.856, 172412.828, 176444.093)
# Customer area bbox: (172161.734, 174265.856, 173412.828, 175444.093)
bbox_customer = (172161.734, 174265.856, 173412.828, 175444.093)

# Visualize sample scenario
plot_scenario_map(sample_data, network_gdf, bbox=bbox_vis, figsize=(10, 10))

In [ ]:
# Compare clustered vs dispersed distributions for inside depot
compare_distributions(leuven_output_path, depot_location='inside', penalty='0.0000', 
                      method='approx_shapley_mc', network_gdf=network_gdf, bbox=bbox_vis)

# Aggregate Analysis Functions

In [ ]:
# =============================================================================
# Aggregate Analysis Functions
# =============================================================================

def collect_all_scenarios_summary(output_path, network_gdf):
    """
    Collect summary statistics for all scenarios.
    
    Returns:
    --------
    DataFrame with scenario info and summary statistics
    """
    all_folders = [f for f in os.listdir(output_path) if os.path.isdir(os.path.join(output_path, f))]
    
    summaries = []
    for folder in all_folders:
        try:
            folder_path = os.path.join(output_path, folder)
            info = parse_leuven_folder_name(folder)
            
            # Read receivers
            receiver_file = os.path.join(folder_path, 'receivers.xml.gz')
            receiver_df = read_receivers_from_xml(receiver_file)
            
            # Read carriers
            carrier_file = os.path.join(folder_path, 'carriers.xml.gz')
            carrier_df = read_carriers_from_xml(carrier_file)
            
            # Calculate summary stats
            total_receivers = len(receiver_df)
            collab_receivers = receiver_df['collaboration_status'].sum()
            non_collab_receivers = total_receivers - collab_receivers
            collab_rate = collab_receivers / total_receivers if total_receivers > 0 else 0
            
            # Score statistics (exclude -100 initial scores)
            valid_scores = receiver_df[receiver_df['score'] > -100]['score']
            mean_score = valid_scores.mean() if len(valid_scores) > 0 else np.nan
            min_score = valid_scores.min() if len(valid_scores) > 0 else np.nan
            max_score = valid_scores.max() if len(valid_scores) > 0 else np.nan
            
            # Carrier stats
            total_carriers = len(carrier_df)
            total_shipments = carrier_df['num_shipments'].sum()
            total_jsprit_cost = carrier_df['jsprit_score'].sum()
            
            summaries.append({
                'folder': folder,
                'depot_location': info['depot_location'],
                'distribution': info['distribution'],
                'method': info['method'],
                'af': float(info['af']) if info['af'] else np.nan,
                'penalty': float(info['penalty']) if info['penalty'] else np.nan,
                'instance': info['instance'],
                'total_receivers': total_receivers,
                'collab_receivers': collab_receivers,
                'non_collab_receivers': non_collab_receivers,
                'collab_rate': collab_rate,
                'mean_receiver_score': mean_score,
                'min_receiver_score': min_score,
                'max_receiver_score': max_score,
                'total_carriers': total_carriers,
                'total_shipments': total_shipments,
                'total_jsprit_cost': total_jsprit_cost
            })
            
        except Exception as e:
            print(f"Error processing {folder}: {e}")
            continue
    
    return pd.DataFrame(summaries)


def count_collaborative_receivers_by_carrier(receiver_df):
    """
    Count how many receivers are collaborative for each affiliated carrier.
    """
    return receiver_df.groupby('affiliated_carrier_id').agg({
        'receiver_id': 'count',
        'collaboration_status': 'sum',
        'score': lambda x: x[x > -100].mean()
    }).rename(columns={
        'receiver_id': 'total_receivers',
        'collaboration_status': 'collab_receivers',
        'score': 'mean_score'
    }).reset_index()


def analyze_collaboration_by_penalty(summary_df):
    """
    Analyze how collaboration rate changes with penalty value.
    """
    return summary_df.groupby(['depot_location', 'distribution', 'method', 'penalty']).agg({
        'collab_rate': 'mean',
        'mean_receiver_score': 'mean',
        'total_jsprit_cost': 'mean'
    }).reset_index()


print("Aggregate analysis functions defined.")

In [ ]:
# Collect summary for all scenarios (this may take a few minutes)
print("Collecting all scenario summaries...")
leuven_summary_df = collect_all_scenarios_summary(leuven_output_path, network_gdf)
print(f"Collected {len(leuven_summary_df)} scenarios")
leuven_summary_df.head(10)

In [ ]:
# Check unique values
print("Depot locations:", leuven_summary_df['depot_location'].unique())
print("Distributions:", leuven_summary_df['distribution'].unique())
print("Methods:", leuven_summary_df['method'].unique())
print("Penalties:", sorted(leuven_summary_df['penalty'].unique()))

# Collaboration Rate Analysis Plots

In [ ]:
# Plot collaboration rate vs penalty for different methods
def plot_collab_rate_vs_penalty(summary_df, depot_location='inside', distribution='clustered', 
                                 figsize=(12, 6)):
    """
    Plot collaboration rate vs penalty for different allocation methods.
    """
    subset = summary_df[(summary_df['depot_location'] == depot_location) & 
                        (summary_df['distribution'] == distribution)]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    methods = subset['method'].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(methods)))
    
    for method, color in zip(methods, colors):
        method_data = subset[subset['method'] == method].sort_values('penalty')
        ax.plot(method_data['penalty'], method_data['collab_rate'], 
                marker='o', label=method, color=color, linewidth=2, markersize=8)
    
    ax.set_xlabel('Penalty Value', fontsize=12)
    ax.set_ylabel('Collaboration Rate', fontsize=12)
    ax.set_title(f'Collaboration Rate vs Penalty\n{depot_location.title()} Depot | {distribution.title()} Distribution', 
                 fontsize=14)
    ax.legend(title='Allocation Method', loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    
    plt.tight_layout()
    plt.show()
    return fig

# Plot for inside depot, clustered distribution
plot_collab_rate_vs_penalty(leuven_summary_df, depot_location='inside', distribution='clustered')

In [ ]:
# Plot for inside depot, dispersed distribution
plot_collab_rate_vs_penalty(leuven_summary_df, depot_location='inside', distribution='dispersed')

In [ ]:
# Compare inside vs outside depot locations
def plot_depot_comparison(summary_df, distribution='clustered', method='approx_shapley_mc', 
                          figsize=(12, 6)):
    """
    Compare collaboration rate between inside and outside depot locations.
    """
    subset = summary_df[(summary_df['distribution'] == distribution) & 
                        (summary_df['method'] == method)]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    for depot_loc, color, marker in [('inside', 'blue', 'o'), ('outside', 'red', 's')]:
        data = subset[subset['depot_location'] == depot_loc].sort_values('penalty')
        if len(data) > 0:
            ax.plot(data['penalty'], data['collab_rate'], 
                    marker=marker, label=f'{depot_loc.title()} Depot', 
                    color=color, linewidth=2, markersize=8)
    
    ax.set_xlabel('Penalty Value', fontsize=12)
    ax.set_ylabel('Collaboration Rate', fontsize=12)
    ax.set_title(f'Collaboration Rate: Inside vs Outside Depot\n{distribution.title()} | {method}', 
                 fontsize=14)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    
    plt.tight_layout()
    plt.show()
    return fig

# Compare depot locations
plot_depot_comparison(leuven_summary_df, distribution='clustered', method='approx_shapley_mc')

# Heatmap: Collaboration Rate by Depot Location and Distribution

In [ ]:
# Create pivot table for heatmap
def create_collab_heatmap(summary_df, method='approx_shapley_mc', figsize=(10, 6)):
    """
    Create a heatmap showing collaboration rate across depot locations and distributions.
    """
    subset = summary_df[summary_df['method'] == method]
    
    # Create pivot table
    pivot = subset.pivot_table(
        values='collab_rate',
        index=['depot_location', 'distribution'],
        columns='penalty',
        aggfunc='mean'
    )
    
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, 
                vmin=0, vmax=1, cbar_kws={'label': 'Collaboration Rate'})
    
    ax.set_title(f'Collaboration Rate Heatmap\nMethod: {method}', fontsize=14)
    ax.set_xlabel('Penalty Value')
    ax.set_ylabel('Depot Location / Distribution')
    
    plt.tight_layout()
    plt.show()
    return fig

create_collab_heatmap(leuven_summary_df, method='approx_shapley_mc')

# Receiver Score Analysis

In [ ]:
# Plot receiver scores distribution for a sample scenario
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
valid_scores = sample_data['receiver_df'][sample_data['receiver_df']['score'] > -100]['score']
axes[0].hist(valid_scores, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Receiver Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Receiver Score Distribution')
axes[0].axvline(valid_scores.mean(), color='red', linestyle='--', label=f'Mean: {valid_scores.mean():.2f}')
axes[0].legend()

# Score by collaboration status
receiver_df = sample_data['receiver_df']
collab_scores = receiver_df[receiver_df['collaboration_status'] == True]['score']
non_collab_scores = receiver_df[receiver_df['collaboration_status'] == False]['score']

data_to_plot = [collab_scores[collab_scores > -100], non_collab_scores[non_collab_scores > -100]]
labels = ['Collaborative', 'Non-Collaborative']
bp = axes[1].boxplot([d for d in data_to_plot if len(d) > 0], labels=[l for l, d in zip(labels, data_to_plot) if len(d) > 0])
axes[1].set_ylabel('Receiver Score')
axes[1].set_title('Score by Collaboration Status')

plt.suptitle(f"Sample Scenario: {sample_data['scenario_info']['distribution']} | p={sample_data['scenario_info']['penalty']}", 
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Carrier Analysis

In [ ]:
# Display carrier information
carrier_df = sample_data['carrier_df']
print(f"Number of carriers: {len(carrier_df)}")
print(f"Total shipments: {carrier_df['num_shipments'].sum()}")
print(f"Total jsprit cost: {carrier_df['jsprit_score'].sum():.2f}")
carrier_df

In [ ]:
# Analyze collaboration by affiliated carrier
collab_by_carrier = count_collaborative_receivers_by_carrier(sample_data['receiver_df'])
collab_by_carrier['collab_rate'] = collab_by_carrier['collab_receivers'] / collab_by_carrier['total_receivers']
collab_by_carrier

# Carrier Cost Analysis Across Scenarios

In [ ]:
# Plot total jsprit cost vs penalty
def plot_carrier_cost_vs_penalty(summary_df, depot_location='inside', distribution='clustered', 
                                  figsize=(12, 6)):
    """
    Plot total jsprit cost vs penalty for different allocation methods.
    """
    subset = summary_df[(summary_df['depot_location'] == depot_location) & 
                        (summary_df['distribution'] == distribution)]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    methods = subset['method'].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(methods)))
    
    for method, color in zip(methods, colors):
        method_data = subset[subset['method'] == method].sort_values('penalty')
        ax.plot(method_data['penalty'], -method_data['total_jsprit_cost'],  # Negate to show as positive cost
                marker='o', label=method, color=color, linewidth=2, markersize=8)
    
    ax.set_xlabel('Penalty Value', fontsize=12)
    ax.set_ylabel('Total Carrier Cost (positive)', fontsize=12)
    ax.set_title(f'Carrier Cost vs Penalty\n{depot_location.title()} Depot | {distribution.title()} Distribution', 
                 fontsize=14)
    ax.legend(title='Allocation Method', loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    return fig

plot_carrier_cost_vs_penalty(leuven_summary_df, depot_location='inside', distribution='clustered')

# Save Summary Data to CSV

In [ ]:
# Save the summary dataframe for later use
leuven_summary_df.to_csv('output/leuven_scenario_summary.csv', index=False)
print(f"Saved summary to output/leuven_scenario_summary.csv")
leuven_summary_df